# STRAT-412: Whole Foods Market Store Directory Scraper

This notebook scrapes the entire Whole Foods Market store directory and exports the data to a CSV file.

**Important:** Whole Foods uses a fully JavaScript-rendered single-page app. Cloudscraper cannot access store data (returns only the HTML shell). This notebook uses the **Whole Foods internal store locator API** to pull all store data directly as JSON — no page scraping needed.

**Output CSV columns:** Store Name, Store Number, Store Complex, Address, City, State, Zip, Phone Number

## Imports & Setup
Install required packages. We use `requests` and `cloudscraper` to hit the Whole Foods internal API directly. Selenium is included as a backup but should not be needed.

In [ ]:
# ─────────────────────────────────────────────
# CELL 0: Install & Setup
# ─────────────────────────────────────────────
!pip install cloudscraper beautifulsoup4 lxml requests --quiet

import cloudscraper
import requests
from bs4 import BeautifulSoup
import csv
import json
import time
import re

# Base URL
BASE_URL = "https://www.wholefoodsmarket.com"

# Create a cloudscraper session (handles Cloudflare challenges)
scraper = cloudscraper.create_scraper(
    browser={
        "browser": "chrome",
        "platform": "windows",
        "desktop": True,
    }
)

# US states — we search each state to collect all stores from the API
US_STATES = [
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA",
    "HI","ID","IL","IN","IA","KS","KY","LA","ME","MD",
    "MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ",
    "NM","NY","NC","ND","OH","OK","OR","PA","RI","SC",
    "SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC"
]

# State abbreviation to full name mapping (for API search queries)
STATE_NAMES = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming", "DC": "District of Columbia"
}

print("Setup complete. Will search " + str(len(US_STATES)) + " states for Whole Foods stores.")

## Discovery: Test the Whole Foods Store API
The Whole Foods website is a JavaScript single-page app — cloudscraper only returns the HTML shell (no store data). Instead, we probe the internal API that the store locator calls. We try several known endpoints to find one that returns JSON store data.

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 (Discovery): Probe for the WFM Store API
# ─────────────────────────────────────────────
# The WFM store locator at /stores is JS-rendered. The JavaScript calls an
# internal API to get store data. We try several known endpoint patterns.

API_URL = None  # will be set once we find a working endpoint
API_MODE = None  # "json_api" or "selenium"

# Headers that mimic a browser XHR request
api_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": BASE_URL + "/stores",
    "Origin": BASE_URL,
}

# List of API endpoint patterns to try
api_candidates = [
    BASE_URL + "/stores/store/query?limit=600",
    BASE_URL + "/stores/store-locator-api/stores?limit=600",
    BASE_URL + "/api/stores?limit=600",
    BASE_URL + "/stores/store-locator?text=California&limit=50",
    BASE_URL + "/stores/store/query?text=California&limit=50",
]

print("Probing for Whole Foods store API...")
print("=" * 60)

for candidate_url in api_candidates:
    try:
        print("\nTrying: " + candidate_url)
        resp = scraper.get(candidate_url, headers=api_headers, timeout=15)
        print("  Status: " + str(resp.status_code))

        # Check if we got JSON back
        content_type = resp.headers.get("Content-Type", "")
        print("  Content-Type: " + content_type)

        if resp.status_code == 200 and "json" in content_type.lower():
            data = resp.json()
            print("  Got JSON response!")
            if isinstance(data, list):
                print("  Array with " + str(len(data)) + " items")
                if len(data) > 0:
                    print("  First item keys: " + str(list(data[0].keys())[:10]))
                    API_URL = candidate_url
                    API_MODE = "json_api"
                    break
            elif isinstance(data, dict):
                print("  Dict keys: " + str(list(data.keys())[:10]))
                # Check for nested store lists
                for key in ["stores", "results", "data", "items"]:
                    if key in data and isinstance(data[key], list) and len(data[key]) > 0:
                        print("  Found '" + key + "' with " + str(len(data[key])) + " items")
                        print("  First item keys: " + str(list(data[key][0].keys())[:10]))
                        API_URL = candidate_url
                        API_MODE = "json_api"
                        break
                if API_MODE:
                    break
        elif resp.status_code == 200:
            # Maybe JSON disguised as text/html?
            try:
                data = resp.json()
                print("  Parsed as JSON despite Content-Type")
                print("  Type: " + str(type(data)))
                if isinstance(data, (list, dict)):
                    API_URL = candidate_url
                    API_MODE = "json_api"
                    break
            except Exception:
                text_preview = resp.text[:300]
                print("  Not JSON. Preview: " + text_preview[:200])
    except Exception as e:
        print("  Error: " + str(e))

print("\n" + "=" * 60)
if API_MODE == "json_api":
    print("SUCCESS: Found working API at:")
    print("  " + API_URL)
else:
    print("No direct API found. Will use Selenium to render JS pages.")
    API_MODE = "selenium"

## Code Block #1: Scrape Location Info for One Store

**Extraction strategy (in order of reliability):**
1. **API JSON** — If the store API works, parse the JSON fields directly (no HTML scraping needed)
2. **Selenium** — If the API fails, use headless Chrome to render the JS page, then extract via JSON-LD / itemprop / regex

In [ ]:
# ─────────────────────────────────────────────
# CODE BLOCK #1: Scrape One Store
# ─────────────────────────────────────────────
# Two methods depending on which approach works:
# Method A: Parse a store dict from the API JSON response
# Method B: Use Selenium to render an individual store page

# --- Selenium setup (only used if API method fails) ---
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)

driver = None  # initialized only if Selenium is needed


def get_soup_selenium(url, wait_seconds=5):
    """Load a page with Selenium and return BeautifulSoup."""
    global driver
    if driver is None:
        driver = webdriver.Chrome(options=chrome_options)
    driver.get(url)
    time.sleep(wait_seconds)
    return BeautifulSoup(driver.page_source, "lxml")


def parse_store_from_api(store_json):
    """
    Parse a single store record from the Whole Foods API JSON.
    The API returns store objects with various field names depending on
    the endpoint. This function handles common field name patterns.
    """
    store_name = ""
    store_number = ""
    store_complex = ""
    address = ""
    city = ""
    state = ""
    zipcode = ""
    phone = ""

    # Store name — try common key patterns
    for key in ["name", "storeName", "store_name", "title"]:
        if key in store_json and store_json[key]:
            store_name = str(store_json[key]).strip()
            break

    # Store number
    for key in ["storeNumber", "store_number", "storeId", "store_id", "branchCode", "id"]:
        if key in store_json and store_json[key]:
            store_number = str(store_json[key]).strip()
            break

    # Address fields — check for nested address object first
    addr_obj = store_json.get("address", store_json.get("location", store_json))
    if isinstance(addr_obj, dict):
        for key in ["streetAddress", "street", "address1", "addressLine1", "line1"]:
            if key in addr_obj and addr_obj[key]:
                address = str(addr_obj[key]).strip()
                break
        for key in ["addressLocality", "city", "locality"]:
            if key in addr_obj and addr_obj[key]:
                city = str(addr_obj[key]).strip()
                break
        for key in ["addressRegion", "state", "region", "stateProvince"]:
            if key in addr_obj and addr_obj[key]:
                state = str(addr_obj[key]).strip()
                break
        for key in ["postalCode", "zip", "zipCode", "postal_code"]:
            if key in addr_obj and addr_obj[key]:
                zipcode = str(addr_obj[key]).strip()
                break

    # If address fields are at the top level (not nested)
    if not address:
        for key in ["streetAddress", "street", "address1", "addressLine1"]:
            if key in store_json and store_json[key]:
                address = str(store_json[key]).strip()
                break
    if not city:
        for key in ["city", "addressLocality", "locality"]:
            if key in store_json and store_json[key]:
                city = str(store_json[key]).strip()
                break
    if not state:
        for key in ["state", "addressRegion", "region", "stateProvince"]:
            if key in store_json and store_json[key]:
                state = str(store_json[key]).strip()
                break
    if not zipcode:
        for key in ["postalCode", "zip", "zipCode", "postal_code"]:
            if key in store_json and store_json[key]:
                zipcode = str(store_json[key]).strip()
                break

    # Phone
    for key in ["telephone", "phone", "phoneNumber", "phone_number"]:
        if key in store_json and store_json[key]:
            phone = str(store_json[key]).strip()
            break

    # Clean up
    zip_match = re.search(r"\d{5}", zipcode)
    if zip_match:
        zipcode = zip_match.group(0)
    state = state.strip().upper()[:2]

    return {
        "Store Name": store_name,
        "Store Number": store_number,
        "Store Complex": store_complex,
        "Address": address,
        "City": city,
        "State": state,
        "Zip": zipcode,
        "Phone Number": phone,
    }


def scrape_one_store_selenium(url):
    """
    Scrape a single store page using Selenium (for when the API fails).
    """
    soup = get_soup_selenium(url, wait_seconds=6)

    store_name = ""
    store_number = ""
    store_complex = ""
    address = ""
    city = ""
    state = ""
    zipcode = ""
    phone = ""

    # Try JSON-LD structured data first
    json_ld_scripts = soup.find_all("script", type="application/ld+json")
    for script in json_ld_scripts:
        try:
            data = json.loads(script.string)
            if isinstance(data, list):
                data = data[0]
            if "address" in data:
                addr_data = data["address"]
                address = addr_data.get("streetAddress", "")
                city = addr_data.get("addressLocality", "")
                state = addr_data.get("addressRegion", "")
                zipcode = addr_data.get("postalCode", "")
            if "telephone" in data:
                phone = data["telephone"]
            if "name" in data:
                store_name = data["name"]
            if "branchCode" in data:
                store_number = data["branchCode"]
        except (json.JSONDecodeError, TypeError, KeyError):
            continue

    # Fallbacks: h1, itemprop, address tag, regex
    if not store_name:
        h1 = soup.find("h1")
        if h1:
            store_name = h1.get_text(strip=True)

    if not store_number:
        page_text = soup.get_text()
        m = re.search(r"Store\s*#?\s*(\d+)", page_text)
        if m:
            store_number = m.group(1)

    if not address:
        el = soup.find(attrs={"itemprop": "streetAddress"})
        if el:
            address = el.get_text(strip=True)
    if not city:
        el = soup.find(attrs={"itemprop": "addressLocality"})
        if el:
            city = el.get_text(strip=True)
    if not state:
        el = soup.find(attrs={"itemprop": "addressRegion"})
        if el:
            state = el.get_text(strip=True)
    if not zipcode:
        el = soup.find(attrs={"itemprop": "postalCode"})
        if el:
            zipcode = el.get_text(strip=True)

    if not phone:
        el = soup.find(attrs={"itemprop": "telephone"})
        if el:
            phone = el.get_text(strip=True)
    if not phone:
        tel_link = soup.find("a", href=re.compile(r"^tel:"))
        if tel_link:
            phone = tel_link.get_text(strip=True) or tel_link["href"].replace("tel:", "").strip()
    if not phone:
        m = re.search(r"\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}", soup.get_text())
        if m:
            phone = m.group(0)

    # Data cleaning
    zip_match = re.search(r"\d{5}", zipcode)
    if zip_match:
        zipcode = zip_match.group(0)
    state = state.strip().upper()[:2]

    return {
        "Store Name": store_name,
        "Store Number": store_number,
        "Store Complex": store_complex,
        "Address": address,
        "City": city,
        "State": state,
        "Zip": zipcode,
        "Phone Number": phone,
    }


# ── Test with one store ────────────────────────────────────────────────────────
print("=" * 60)
print("SCRAPING ONE STORE")
print("=" * 60)

if API_MODE == "json_api":
    print("Using API mode — testing by fetching one state...")
    test_api_url = API_URL.split("?")[0] + "?text=Arlington+VA&limit=5"
    try:
        resp = scraper.get(test_api_url, headers=api_headers, timeout=15)
        data = resp.json()
        # Navigate to the store list
        stores_list = data if isinstance(data, list) else data.get("stores", data.get("results", data.get("data", [])))
        if stores_list and len(stores_list) > 0:
            result = parse_store_from_api(stores_list[0])
            print("API returned store:")
            for key, value in result.items():
                print("  " + key + ": " + value)
            print("\nRaw JSON keys: " + str(list(stores_list[0].keys())))
        else:
            print("API returned empty list. Switching to Selenium mode.")
            API_MODE = "selenium"
    except Exception as e:
        print("API test failed: " + str(e))
        print("Switching to Selenium mode.")
        API_MODE = "selenium"

if API_MODE == "selenium":
    print("Using Selenium mode — loading store page with headless Chrome...")
    one_store_url = BASE_URL + "/stores/arlington"
    result = scrape_one_store_selenium(one_store_url)
    for key, value in result.items():
        print("  " + key + ": " + value)
    if not result["Address"]:
        print("\nWARNING: Selenium also returned no address data.")
        print("The Whole Foods website may be blocking automated access.")
        print("Try running again or check if Chrome/ChromeDriver versions match.")

print("\nActive mode: " + API_MODE)

## Code Block #2: Test with a Small Loop of 3-4 Stores
Test the scrape function on a few stores to verify data quality before the full run.

In [ ]:
# ─────────────────────────────────────────────
# CODE BLOCK #2: Test Small Loop (3-4 stores)
# ─────────────────────────────────────────────

print("=" * 60)
print("TESTING WITH SMALL LOOP")
print("=" * 60)

test_results = []

if API_MODE == "json_api":
    # Test by querying a few different search terms
    test_queries = ["Arlington VA", "Austin TX", "Brooklyn NY", "Chicago IL"]
    for query in test_queries:
        print("\nQuerying API for: " + query)
        try:
            test_url = API_URL.split("?")[0] + "?text=" + query.replace(" ", "+") + "&limit=2"
            resp = scraper.get(test_url, headers=api_headers, timeout=15)
            data = resp.json()
            stores_list = data if isinstance(data, list) else data.get("stores", data.get("results", data.get("data", [])))
            if stores_list and len(stores_list) > 0:
                result = parse_store_from_api(stores_list[0])
                test_results.append(result)
                for key, value in result.items():
                    print("  " + key + ": " + value)
            else:
                print("  No results for this query")
        except Exception as e:
            print("  Error: " + str(e))
        time.sleep(1)

elif API_MODE == "selenium":
    test_slugs = ["arlington", "austin-domain", "brooklyn-third-and-third", "chicago-lakeview"]
    for slug in test_slugs:
        url = BASE_URL + "/stores/" + slug
        print("\nScraping: " + url)
        try:
            result = scrape_one_store_selenium(url)
            test_results.append(result)
            for key, value in result.items():
                print("  " + key + ": " + value)
        except Exception as e:
            print("  ERROR: " + str(e))

print("\nSuccessfully scraped " + str(len(test_results)) + " test stores.")

## Code Block #3: Print the List of State URLs / State Queries
Whole Foods does not have state-level directory pages. Instead, we query the API (or Selenium) for each US state to discover all stores.

In [ ]:
# ─────────────────────────────────────────────
# CODE BLOCK #3: Print the State Search Queries
# ─────────────────────────────────────────────
# Whole Foods has no state-level directory pages like Sprouts or TJ's.
# We search the API by state name to collect all stores nationwide.

print("=" * 60)
print("STATE SEARCH QUERIES")
print("=" * 60)
print("Whole Foods does not have state-level directory pages.")
print("We will search by each state to collect all stores.\n")

state_queries = []
for abbr in US_STATES:
    full_name = STATE_NAMES.get(abbr, abbr)
    state_queries.append((abbr, full_name))
    print("  " + abbr + " -> Search: \"" + full_name + "\"")

print("\nTotal state queries: " + str(len(state_queries)))

## Code Block #4: Collect All Store Data by State
Query the API (or use Selenium) for each state. Deduplicate stores by store number/name to build the complete list.

In [ ]:
# ─────────────────────────────────────────────
# CODE BLOCK #4: Collect All Store Data by State
# ─────────────────────────────────────────────
# For API mode: query each state name, collect all stores, deduplicate.
# For Selenium mode: use Selenium to search the store locator by state.

print("=" * 60)
print("COLLECTING ALL STORES BY STATE")
print("=" * 60)

all_stores_raw = []  # list of store dicts
seen_keys = set()    # for deduplication

if API_MODE == "json_api":
    # ── API approach: query each state ──
    for i, (abbr, full_name) in enumerate(state_queries, start=1):
        query_url = API_URL.split("?")[0] + "?text=" + full_name.replace(" ", "+") + "&limit=200"
        try:
            resp = scraper.get(query_url, headers=api_headers, timeout=15)
            data = resp.json()
            stores_list = data if isinstance(data, list) else data.get("stores", data.get("results", data.get("data", [])))

            state_count = 0
            if stores_list:
                for store_json in stores_list:
                    parsed = parse_store_from_api(store_json)
                    # Dedup key: store number + name + zip
                    dedup_key = parsed["Store Number"] + "|" + parsed["Store Name"] + "|" + parsed["Zip"]
                    if dedup_key not in seen_keys:
                        seen_keys.add(dedup_key)
                        all_stores_raw.append(parsed)
                        state_count += 1

            print("  " + abbr + " (" + full_name + "): " + str(state_count) + " new stores (total: " + str(len(all_stores_raw)) + ")")
        except Exception as e:
            print("  " + abbr + " ERROR: " + str(e))
        time.sleep(1)

elif API_MODE == "selenium":
    # ── Selenium approach: use the store locator search ──
    print("Using Selenium to search the Whole Foods store locator by state...")
    print("This will take several minutes.\n")

    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.keys import Keys

    if driver is None:
        driver = webdriver.Chrome(options=chrome_options)

    for i, (abbr, full_name) in enumerate(state_queries, start=1):
        try:
            # Load the store locator page
            driver.get(BASE_URL + "/stores")
            time.sleep(4)

            # Try to find the search input and type the state name
            search_inputs = driver.find_elements(By.CSS_SELECTOR, "input[type='search'], input[type='text'], input[placeholder*='search'], input[placeholder*='zip'], input[placeholder*='city'], input[aria-label*='search']")
            if search_inputs:
                search_input = search_inputs[0]
                search_input.clear()
                search_input.send_keys(full_name)
                search_input.send_keys(Keys.RETURN)
                time.sleep(5)

                # Parse the results page
                soup = BeautifulSoup(driver.page_source, "lxml")

                # Look for store links in the results
                store_links = []
                for link in soup.find_all("a", href=True):
                    href = link["href"]
                    if re.match(r"^/stores/[a-z][\w-]+$", href):
                        slug = href.split("/stores/")[-1]
                        if slug and not any(x in slug for x in ["new-store", "site-map", "search", "find"]):
                            full_url = BASE_URL + href
                            if full_url not in seen_keys:
                                seen_keys.add(full_url)
                                store_links.append(full_url)

                state_count = 0
                for url in store_links:
                    try:
                        result = scrape_one_store_selenium(url)
                        all_stores_raw.append(result)
                        state_count += 1
                    except Exception as e:
                        print("    Error on " + url + ": " + str(e))

                print("  " + abbr + ": " + str(state_count) + " stores (total: " + str(len(all_stores_raw)) + ")")
            else:
                print("  " + abbr + ": Could not find search input on page")
        except Exception as e:
            print("  " + abbr + " ERROR: " + str(e))

print("\n" + "=" * 60)
print("Total unique stores collected: " + str(len(all_stores_raw)))

## Code Block #5: Print All Store Records
Display all collected stores to verify data quality.

In [ ]:
# ─────────────────────────────────────────────
# CODE BLOCK #5: Print All Store Records
# ─────────────────────────────────────────────

print("=" * 60)
print("ALL COLLECTED STORES: " + str(len(all_stores_raw)))
print("=" * 60)

# Sort by state then city
all_stores_sorted = sorted(all_stores_raw, key=lambda x: (x["State"], x["City"], x["Store Name"]))

for i, store in enumerate(all_stores_sorted, start=1):
    print(str(i) + ". " + store["Store Name"]
          + " | #" + store["Store Number"]
          + " | " + store["Address"]
          + ", " + store["City"]
          + ", " + store["State"]
          + " " + store["Zip"]
          + " | " + store["Phone Number"])

# Count by state
print("\n" + "=" * 60)
print("STORES BY STATE")
print("=" * 60)
from collections import Counter
state_counts = Counter(s["State"] for s in all_stores_sorted if s["State"])
for state_abbr, count in sorted(state_counts.items()):
    print("  " + state_abbr + ": " + str(count) + " stores")
print("\nTotal states: " + str(len(state_counts)))

## Code Block #6: Full Scraping Loop (validation pass)
If using the API, data was already collected in Code Block #4. This cell validates the data and fills in any missing fields.

If using Selenium, this is where we'd do a second pass on any stores with missing data.

In [ ]:
# ─────────────────────────────────────────────
# CODE BLOCK #6: Validation Pass
# ─────────────────────────────────────────────
# Check data quality. Flag any stores with missing critical fields.

print("=" * 60)
print("DATA VALIDATION")
print("=" * 60)

all_stores = all_stores_sorted  # use the sorted list from above

# Check for missing fields
missing_name    = sum(1 for s in all_stores if not s["Store Name"])
missing_address = sum(1 for s in all_stores if not s["Address"])
missing_city    = sum(1 for s in all_stores if not s["City"])
missing_state   = sum(1 for s in all_stores if not s["State"])
missing_zip     = sum(1 for s in all_stores if not s["Zip"])
missing_phone   = sum(1 for s in all_stores if not s["Phone Number"])

print("Total stores: " + str(len(all_stores)))
print("Missing Store Name:    " + str(missing_name))
print("Missing Address:       " + str(missing_address))
print("Missing City:          " + str(missing_city))
print("Missing State:         " + str(missing_state))
print("Missing Zip:           " + str(missing_zip))
print("Missing Phone:         " + str(missing_phone))

# Show stores with missing address (the most critical field)
if missing_address > 0:
    print("\nStores missing address:")
    for s in all_stores:
        if not s["Address"]:
            print("  " + s["Store Name"] + " (State: " + s["State"] + ")")

print("\nValidation complete.")

## Code Block #7: Write to CSV and Export
Write the collected store data to a CSV file and download it.

In [ ]:
# ─────────────────────────────────────────────
# CODE BLOCK #7: Write to CSV and Export
# ─────────────────────────────────────────────

csv_filename = "whole_foods_stores.csv"
csv_columns = [
    "Store Name",
    "Store Number",
    "Store Complex",
    "Address",
    "City",
    "State",
    "Zip",
    "Phone Number",
]

with open(csv_filename, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(all_stores)

print("CSV file '" + csv_filename + "' written with " + str(len(all_stores)) + " rows.")
print("Columns: " + ", ".join(csv_columns))

print("\nPreview (first 10 rows):")
for store in all_stores[:10]:
    print("  " + store["Store Name"]
          + " | #" + store["Store Number"]
          + " | " + store["Address"]
          + ", " + store["City"]
          + ", " + store["State"]
          + " " + store["Zip"]
          + " | " + store["Phone Number"])

# Clean up Selenium driver if it was used
if driver is not None:
    try:
        driver.quit()
        print("\nSelenium driver closed.")
    except Exception:
        pass

# Download the CSV in Google Colab
from google.colab import files
files.download(csv_filename)